In [ ]:
from collections import defaultdict

# Step 1: Read Corpus
def read_corpus():
    print("Enter tagged corpus (word/TAG). Type 'END' to stop:")
    corpus = []
    while True:
        line = input()
        if line == "END":
            break
        corpus.append(line.strip())
    return corpus


# Step 2: Train HMM
def train_hmm(corpus):
    transition = defaultdict(lambda: defaultdict(int))
    emission = defaultdict(lambda: defaultdict(int))
    tag_count = defaultdict(int)

    START = "<START>"

    for line in corpus:
        tokens = line.split()
        prev_tag = START

        for token in tokens:
            word, tag = token.rsplit("/", 1)

            emission[tag][word] += 1
            transition[prev_tag][tag] += 1
            tag_count[tag] += 1

            prev_tag = tag

    # Convert to probabilities
    transition_prob = defaultdict(dict)
    emission_prob = defaultdict(dict)

    for prev in transition:
        total = sum(transition[prev].values())
        for curr in transition[prev]:
            transition_prob[prev][curr] = transition[prev][curr] / total

    for tag in emission:
        total = sum(emission[tag].values())
        for word in emission[tag]:
            emission_prob[tag][word] = emission[tag][word] / total

    return transition_prob, emission_prob


# Step 3: Compute Probability for Given Sequence
def compute_prob(sequence, words, transition_prob, emission_prob):
    prob = 1.0
    prev_tag = "<START>"

    for i in range(len(sequence)):
        tag = sequence[i]
        word = words[i]

        trans = transition_prob.get(prev_tag, {}).get(tag, 0)
        emis = emission_prob.get(tag, {}).get(word, 0)

        prob *= trans * emis
        prev_tag = tag

    return prob


# MAIN
corpus = read_corpus()
transition_prob, emission_prob = train_hmm(corpus)

# Show probabilities
print("\nTransition Probabilities:")
for prev in transition_prob:
    for curr in transition_prob[prev]:
        print(f"P({curr}|{prev}) = {transition_prob[prev][curr]:.4f}")

print("\nEmission Probabilities:")
for tag in emission_prob:
    for word in emission_prob[tag]:
        print(f"P({word}|{tag}) = {emission_prob[tag][word]:.4f}")


# Input sentence
sentence = input("\nEnter sentence: ")
words = sentence.split()

# Input number of sequences
n = int(input("Enter number of tag sequences (C1, C2...): "))

sequences = []

print("Enter sequences (space separated tags):")
for i in range(n):
    seq = input(f"C{i+1}: ").split()
    sequences.append(seq)

print("\nSequence Probabilities:\n")

max_prob = -1
best_seq = None

for i, seq in enumerate(sequences):
    prob = compute_prob(seq, words, transition_prob, emission_prob)
    print(f"C{i+1} {seq} => {prob}")

    if prob > max_prob:
        max_prob = prob
        best_seq = seq

# Final Answer
print("\nBest Sequence (Maximum Probability):")
for w, t in zip(words, best_seq):
    print(f"{w} -> {t}")

print(f"\nMax Probability = {max_prob}")

Enter tagged corpus (word/TAG). Type 'END' to stop:
big/Adjective dog/Noun runs/Verb fast/Adverb
dog/Noun eats/Verb food/Noun
happy/Adjective boy/Noun plays/Verb
boy/Noun runs/Verb quickly/Adverb
small/Adjective cat/Noun sleeps/Verb
cat/Noun runs/Verb fast/Adverb
END

Transition Probabilities:
P(Adjective|<START>) = 0.5000
P(Noun|<START>) = 0.5000
P(Noun|Adjective) = 1.0000
P(Verb|Noun) = 1.0000
P(Adverb|Verb) = 0.7500
P(Noun|Verb) = 0.2500

Emission Probabilities:
P(big|Adjective) = 0.3333
P(happy|Adjective) = 0.3333
P(small|Adjective) = 0.3333
P(dog|Noun) = 0.2857
P(food|Noun) = 0.1429
P(boy|Noun) = 0.2857
P(cat|Noun) = 0.2857
P(runs|Verb) = 0.5000
P(eats|Verb) = 0.1667
P(plays|Verb) = 0.1667
P(sleeps|Verb) = 0.1667
P(fast|Adverb) = 0.6667
P(quickly|Adverb) = 0.3333

Enter sentence: dog runs fast
Enter number of tag sequences (C1, C2...): 3
Enter sequences (space separated tags):
C1: Noun Verb Adverb
C2: Adjective Noun Verb
C3: Noun Verb Noun

Sequence Probabilities:

C1 ['Noun', 'Ve